In [1]:
import pandas as pd

from testing.r_and_d.boolean_queries.query_tester import (
    BooleanQueryTester,
    use_baseline_query,
    generate_with_llm,
    query_openalex_minimal,
    reference_df,
)

from testing.r_and_d.boolean_queries.prompts import (
    policy_atlas_v1,
    policy_atlas_v2,
    wang_et_al_q2_prompt,
    wang_et_al_q3_prompt,
)

from app.services.openalex import sanitize_openalex_query
from testing import TESTING_DIR, logger

BOOL_DIR = TESTING_DIR / "r_and_d/boolean_queries/outputs/"
baseline_results = pd.read_json(BOOL_DIR / "baseline_results.jsonl", lines=True)


2025-11-17 17:53:15,414 - app.core.config - INFO - BACKEND_CORS_ORIGINS parsed from comma-separated: ['discoverypolicyatlas-production.up.railway.app', 'http://localhost:3000']


In [4]:
q = reference_df.question.iloc[3]
q

'Does universal breastfeeding information provision improve health outcomes for mothers and infants?'

In [5]:
query = await use_baseline_query(q, "model", 0, "prompt")
query = sanitize_openalex_query(query)
query

'(Breastfeed* OR "Breast feed" OR Breastfeeding* OR Breastfed OR "infant feed*" OR "Formula feed*" OR "infant formula" OR "infant food")  AND  (Information* OR Education* OR Guidance* OR Knowledge* OR "Health Knowledge Attitudes Practice" OR "Information Seeking Behavior")  AND  (obesity OR overweight OR "over-weight" OR BMI OR "body weight" OR bodyweight OR "Body mass index") '

In [6]:
await query_openalex_minimal(query, count_only=True)

3432

## Prompting experiments

In [7]:
test_prompt = """
Transform user input into a high quality boolean query 
for querying the OpenAlex academic research database.

# Guidance from OpenAlex
Imagine you are an expert systematic review information specialist; now you are given a systematic review
research topic, with the topic title provided by the user below. Your task is to generate a highly effective systematic
review Boolean query to search on OpenAlex (refer to the professionally made ones); the query needs to be
as inclusive as possible so that it can retrieve all the relevant studies that can be included in the research
topic; on the other hand, the query needs to retrieve fewer irrelevant studies so that researchers can spend
less time judging the retrieved documents.

# Important instructions

Return ONLY the boolean query string, nothing else.
"""

In [9]:
query = await generate_with_llm(
    question=q,
    model="gpt-4.1",
    temperature=1.0,
    system_prompt=wang_et_al_q3_prompt,
)
logger.info(q)
logger.info("=" * 20)
logger.info(query)
logger.info("=" * 20)
n_results = await query_openalex_minimal(query, count_only=True)
logger.info(f"Number of retrieved results: {n_results}")

2025-11-17 17:44:39,870 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-17 17:44:39,872 - testing - INFO - Does universal breastfeeding information provision improve health outcomes for mothers and infants?
2025-11-17 17:44:39,873 - testing - INFO - ====================
2025-11-17 17:44:39,873 - testing - INFO - ("breastfeed*" OR "breast-feeding" OR "breast feeding" OR "lactation*" OR "breast milk" OR "human milk") AND ("information provision" OR "information" OR "health education" OR "health promotion" OR "counseling" OR "counselling" OR "patient education" OR "education" OR "advice" OR "guidance" OR "support" OR "instruction" OR "awareness") AND ("universal" OR "population-based" OR "public health" OR "community-wide" OR "general population" OR "all mothers" OR "all women") AND ("mother*" OR "maternal" OR "parent*" OR "infant*" OR "newborn*" OR "neonate*" OR "child*" OR "baby" OR "babies")
2025-11-17 17:44:39,873 - testing - IN

In [7]:
async def repeated_queries(
    question: str,
    model: str,
    temperature: float,
    system_prompt: str,
    n_runs: int = 5,
) -> str:
    """Generate a boolean query using the given parameters."""
    dfs = []
    for _ in range(n_runs):
        query = await generate_with_llm(
            question=question,
            model=model,
            temperature=temperature,
            system_prompt=system_prompt,
        )
        df, _ = await query_openalex_minimal(query, count_only=False)
        dfs.append(df.assign(query=query))
    return (
        pd.concat(dfs)
        .drop_duplicates(subset=["id"])
    )
    

In [28]:
q = "what is the impact of financial incentive on parental engagement in evidence based parenting programmes?"

In [30]:
results = await repeated_queries(q, "gpt-4.1", 1.0, wang_et_al_q3_prompt, n_runs=3)
len(results)

2025-11-21 11:23:01,520 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-21 11:23:05,374 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-21 11:23:10,524 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


19

In [32]:
results

,id,doi,title,cited_by_count,relevance_score,query
0,https://openalex.org/W2071247495,https://doi.org/10.1007/s10935-011-0255-7,Cost-Effectiveness of Childcare Discounts on Parent Participation in Preventive Parent Training in Low-Income Communities,53,35.139553,"(""financial incentive*"" OR ""monetary incentive*"" OR ""cash incentive*"" OR ""financial reward*"" OR ""monetary reward*"" OR ""cash transfer*"" OR ""conditional cash"" OR ""financial support"" OR ""monetary sup..."
1,https://openalex.org/W2909125864,https://doi.org/10.1007/s11121-019-0977-y,Financial Incentives for Promoting Participation in a School-Based Parenting Program in Low-Income Communities,35,31.236395,"(""financial incentive*"" OR ""monetary incentive*"" OR ""cash incentive*"" OR ""financial reward*"" OR ""monetary reward*"" OR ""cash transfer*"" OR ""conditional cash"" OR ""financial support"" OR ""monetary sup..."
2,https://openalex.org/W3013825480,https://doi.org/10.2196/15777,Smartphone Self-Monitoring by Young Adolescents and Parents to Assess and Improve Family Functioning: Qualitative Feasibility Study,22,9.457490,"(""financial incentive*"" OR ""monetary incentive*"" OR ""cash incentive*"" OR ""financial reward*"" OR ""monetary reward*"" OR ""cash transfer*"" OR ""conditional cash"" OR ""financial support"" OR ""monetary sup..."
3,https://openalex.org/W2286332016,None,An expression of deep concern : championing the needs of never married couples and their children coping with family related litigation and separation : a project in collaboration with the Hampshi...,0,2.229695,"(""financial incentive*"" OR ""monetary incentive*"" OR ""cash incentive*"" OR ""financial reward*"" OR ""monetary reward*"" OR ""cash transfer*"" OR ""conditional cash"" OR ""financial support"" OR ""monetary sup..."
4,https://openalex.org/W4236433325,https://doi.org/10.2196/preprints.15777,Smartphone Self-Monitoring by Young Adolescents and Parents to Assess and Improve Family Functioning: Qualitative Feasibility Study (Preprint),0,0.830781,"(""financial incentive*"" OR ""monetary incentive*"" OR ""cash incentive*"" OR ""financial reward*"" OR ""monetary reward*"" OR ""cash transfer*"" OR ""conditional cash"" OR ""financial support"" OR ""monetary sup..."
0,https://openalex.org/W2012658896,https://doi.org/10.1080/15374411003691792,Effects of Monetary Incentives on Engagement in the PACE Parenting Program,66,33.804028,"(""financial incentive*"" OR ""monetary incentive*"" OR ""cash transfer*"" OR ""payment*"" OR ""voucher*"" OR ""reward*"" OR ""conditional cash*"" OR ""financial support"" OR ""economic incentive*"" OR ""remuneratio..."
2,https://openalex.org/W1567138243,None,Home-School Relations: Working Successfully With Parents and Families,121,26.049816,"(""financial incentive*"" OR ""monetary incentive*"" OR ""cash transfer*"" OR ""payment*"" OR ""voucher*"" OR ""reward*"" OR ""conditional cash*"" OR ""financial support"" OR ""economic incentive*"" OR ""remuneratio..."
3,https://openalex.org/W2424850770,https://doi.org/10.1177/0165025416652248,Pathways to improved development for children living in poverty,27,23.536121,"(""financial incentive*"" OR ""monetary incentive*"" OR ""cash transfer*"" OR ""payment*"" OR ""voucher*"" OR ""reward*"" OR ""conditional cash*"" OR ""financial support"" OR ""economic incentive*"" OR ""remuneratio..."
4,https://openalex.org/W2972911431,https://doi.org/10.1080/01490400.2019.1646172,“The Credit Card or the Taxi”: A Qualitative Investigation of Parent Involvement in Indoor Competition Climbing,15,16.047722,"(""financial incentive*"" OR ""monetary incentive*"" OR ""cash transfer*"" OR ""payment*"" OR ""voucher*"" OR ""reward*"" OR ""conditional cash*"" OR ""financial support"" OR ""economic incentive*"" OR ""remuneratio..."
5,https://openalex.org/W4412786664,https://doi.org/10.70619/vol5iss3pp32-46,"Effectiveness of Parental Involvement on Students’ Participation in Schooling in Public Secondary Schools in Meru County, Kenya",0,2.110603,"("

In [10]:
pd.set_option("display.max_colwidth", 200)
results.sort_values("relevance_score", ascending=False)[["title", "relevance_score", "query"]]

,title,relevance_score,query
0,Lactation Counseling for Mothers of Very Low Birth Weight Infants: Effect on Maternal Anxiety and Infant Intake of Human Milk,679.863830,"(breastfeed* OR ""breast feeding"" OR ""breast-fed"" OR ""breast fed"" OR lactation OR ""exclusive breastfeeding"" OR nursing OR ""human milk feeding"") AND (information OR education OR counselling OR couns..."
1,The Effect of Postpartum Lactation Counseling on the Duration of Breast-feeding in Low-Income Women,507.266200,"(breastfeed* OR ""breast feeding"" OR ""breast-fed"" OR ""breast fed"" OR lactation OR ""exclusive breastfeeding"" OR nursing OR ""human milk feeding"") AND (information OR education OR counselling OR couns..."
2,Commercial Discharge Packs and Breast-Feeding Counseling: Effects on Infant-Feeding Practices in a Randomized Trial,486.287960,"(breastfeed* OR ""breast feeding"" OR ""breast-fed"" OR ""breast fed"" OR lactation OR ""exclusive breastfeeding"" OR nursing OR ""human milk feeding"") AND (information OR education OR counselling OR couns..."
3,"Postnatal peer counselling on exclusive breastfeeding of low‐birthweight infants: A randomized, controlled trial",388.754180,"(breastfeed* OR ""breast feeding"" OR ""breast-fed"" OR ""breast fed"" OR lactation OR ""exclusive breastfeeding"" OR nursing OR ""human milk feeding"") AND (information OR education OR counselling OR couns..."
4,Counseling About the Maternal Health Benefits of Breastfeeding and Mothers’ Intentions to Breastfeed,386.651430,"(breastfeed* OR ""breast feeding"" OR ""breast-fed"" OR ""breast fed"" OR lactation OR ""exclusive breastfeeding"" OR nursing OR ""human milk feeding"") AND (information OR education OR counselling OR couns..."
...,...,...,...
3801,Editorial introductions,0.312252,"(""breastfeeding"" OR ""breast feeding"" OR ""lactation"" OR ""exclusive breastfeeding"" OR ""exclusive breast feeding"" OR ""breast milk"") AND (""information provision"" OR ""health education"" OR ""information ..."
3802,Editorial introductions,0.298805,"(""breastfeeding"" OR ""breast feeding"" OR ""lactation"" OR ""exclusive breastfeeding"" OR ""exclusive breast feeding"" OR ""breast milk"") AND (""information provision"" OR ""health education"" OR ""information ..."
3803,Guidelines for Comprehensive Prevention and Treatment of Cardiovascular Diseases in Community Populations (The Trial),0.296899,"(""breastfeeding"" OR ""breast feeding"" OR ""lactation"" OR ""exclusive breastfeeding"" OR ""exclusive breast feeding"" OR ""breast milk"") AND (""information provision"" OR ""health education"" OR ""information ..."
3804,NAPNAP update,0.177760,"(""breastfeeding"" OR ""breast feeding"" OR ""lactation"" OR ""exclusive breastfeeding"" OR ""exclusive breast feeding"" OR ""breast milk"") AND (""information provision"" OR ""health education"" OR ""information ..."
